# 🧬 02 — Extract Activations & Find Refusal Direction

**Tujuan:** Ekstrak aktivasi residual stream dari harmful vs harmless prompts, lalu hitung refusal direction vector.

**Lokasi:** ⚠️ **Colab recommended** (butuh cache aktivasi banyak layer). Bisa di VS Code dengan batch kecil.

---
**Reference:** ortho_cookbook.ipynb (FailSpy), abliteration article (mlabonne)

**Key improvements from reference:**
- Harmful + harmless diproses **bersamaan** agar dimensi aligned
- Memory management: `gc.collect(); torch.cuda.empty_cache()` setelah tiap batch
- `defaultdict` untuk collection yang lebih clean
- Batch size lebih kecil untuk 4GB VRAM

**Output:** `models/abliterated/refusal_dirs.safetensors`

Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import sys, os, gc


In [ ]:
import torch
from collections import defaultdict
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer, utils

# Matikan autograd untuk hemat GPU memory (credit: Undi95)
torch.set_grad_enabled(False)

from safetensors.torch import save_file

from src.config import DEFAULT_MODEL, DEVICE, N_INST_TRAIN, BATCH_SIZE_EXTRACT, ACTIVATION_LAYERS, ABLITERATED_DIR, SEED
from src.data_loader import get_harmful_instructions, get_harmless_instructions
from src.generation_utils import tokenize_instructions_fn, clear_memory

torch.manual_seed(SEED)
print(f"Device: {DEVICE}")
print(f"Model: {DEFAULT_MODEL}")
print(f"N_INST_TRAIN: {N_INST_TRAIN}, BATCH_SIZE: {BATCH_SIZE_EXTRACT}")

In [ ]:
# ─── Load model ───
model = HookedTransformer.from_pretrained_no_processing(
    DEFAULT_MODEL,
    dtype=torch.bfloat16,
    default_padding_side='left',
    device=DEVICE,
)
model.tokenizer.padding_side = 'left'
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f"Model loaded. n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}")

In [ ]:
# ─── Load data ───
harmful = get_harmful_instructions()[:N_INST_TRAIN]
harmless = get_harmless_instructions()[:N_INST_TRAIN]
print(f"Harmful: {len(harmful)}, Harmless: {len(harmless)}")

### ⚙️ Ekstraksi Aktivasi (Batch Aligned)

**Penting:** Harmful + harmless diproses dalam 1 batch agar dimensinya aligned.
Cache disimpan di CPU agar VRAM tidak penuh.

Pattern ini dari ortho_cookbook.ipynb (FailSpy).

In [ ]:
# ─── Tokenize dulu semua ───
toks = tokenize_instructions_fn(
    model,
    instructions=harmful + harmless
)
harmful_toks, harmless_toks = toks.split(N_INST_TRAIN)
print(f"Harmful tokens: {harmful_toks.shape}")
print(f"Harmless tokens: {harmless_toks.shape}")

In [ ]:
# ─── Ekstraksi dengan defaultdict pattern ───
harmful = defaultdict(list)
harmless = defaultdict(list)

batch_size = BATCH_SIZE_EXTRACT
num_batches = (N_INST_TRAIN + batch_size - 1) // batch_size

for i in tqdm(range(num_batches)):
    s = i * batch_size
    e = min(N_INST_TRAIN, s + batch_size)

    # Run harmful & harmless dalam batch terpisah, cache di CPU
    harmful_logits, harmful_cache = model.run_with_cache(
        harmful_toks[s:e],
        names_filter=lambda hook_name: 'resid' in hook_name,
        device='cpu',
        reset_hooks_end=True,
    )
    harmless_logits, harmless_cache = model.run_with_cache(
        harmless_toks[s:e],
        names_filter=lambda hook_name: 'resid' in hook_name,
        device='cpu',
        reset_hooks_end=True,
    )

    # Kumpulkan cache
    for key in harmful_cache:
        harmful[key].append(harmful_cache[key])
        harmless[key].append(harmless_cache[key])

    # Flush memory — penting untuk mencegah OOM!
    del harmful_logits, harmless_logits, harmful_cache, harmless_cache
    clear_memory()

# Gabungkan semua batch
harmful = {k: torch.cat(v) for k, v in harmful.items()}
harmless = {k: torch.cat(v) for k, v in harmless.items()}

sample_key = utils.get_act_name('resid_pre', 0)
print(f"Harmful activations: {harmful[sample_key].shape}")
print(f"Harmless activations: {harmless[sample_key].shape}")

### 📐 Hitung Refusal Direction

Formula: `refusal_dir = mean(harmful_activation) - mean(harmless_activation)`, dinormalisasi.

Untuk tiap layer (1 sampai n_layers-1) dan tiap activation layer (pre/mid/post).

In [ ]:
def get_act_idx(cache_dict, act_name, layer):
    key = (act_name, layer)
    return cache_dict[utils.get_act_name(*key)]


activation_refusals = {k: [] for k in ACTIVATION_LAYERS}

for layer_num in range(1, model.cfg.n_layers):
    pos = -1  # token terakhir

    for layer in ACTIVATION_LAYERS:
        harmful_mean = get_act_idx(harmful, layer, layer_num)[:, pos, :].mean(dim=0)
        harmless_mean = get_act_idx(harmless, layer, layer_num)[:, pos, :].mean(dim=0)

        refusal_dir = harmful_mean - harmless_mean
        refusal_dir = refusal_dir / refusal_dir.norm()
        activation_refusals[layer].append(refusal_dir)

print("Refusal directions computed!")
for layer in ACTIVATION_LAYERS:
    print(f"  {layer}: {len(activation_refusals[layer])} directions")

### 💾 Save ke Safetensors

**Kenapa safetensors?** Aman dari pickle-based RCE, standar HuggingFace, loading lebih cepat.
**Plus save .pth cadangan** untuk kompatibilitas dengan notebook referensi ortho_cookbook.

In [ ]:
ABLITERATED_DIR.mkdir(parents=True, exist_ok=True)

# FORMAT UTAMA: Safetensors (aman, standar HF)
# Simpan refusal_dirs sebagai safetensors
# Gabungin semua tensor ke 1 file
refusal_tensors = {}
for layer_name, dir_list in activation_refusals.items():
    for idx, dir_tensor in enumerate(dir_list):
        refusal_tensors[f"{layer_name}.layer_{idx+1}"] = dir_tensor

save_file(refusal_tensors, str(ABLITERATED_DIR / 'refusal_dirs.safetensors'))
print(f"Saved: {ABLITERATED_DIR / 'refusal_dirs.safetensors'}")
print(f"   {len(refusal_tensors)} tensors")


# FORMAT CADANGAN: .pth (kompatibel dengan notebook ortho_cookbook asli)
# Hanya untuk keperluan debugging/referensi — bukan format utama
torch.save(activation_refusals, str(ABLITERATED_DIR / 'refusal_dirs.pth'))
print(f"Backup saved: {ABLITERATED_DIR / 'refusal_dirs.pth'}")

In [ ]:
# ─── Verifikasi ───
from safetensors.torch import load_file
loaded = load_file(str(ABLITERATED_DIR / 'refusal_dirs.safetensors'))
print(f"Verified: {len(loaded)} keys loaded")
for k, v in list(loaded.items())[:3]:
    print(f"  {k}: shape={v.shape}")

In [ ]:
# ─── Cleanup ───
del model, harmful, harmless, harmful_toks, harmless_toks
clear_memory()
print("🧹 Memory cleaned up")

---
**Next:** Lanjut ke `03_screening_directions.ipynb` untuk memilih layer terbaik.